In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import gcsfs
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

In [ ]:
LAT = 32.72
LON = -117.16
START_YEAR = 2000
END_YEAR = 2050
SCENARIO = "ssp585"
PREFERRED_MODEL = "CESM2"

In [ ]:
catalog = pd.read_csv("https://storage.googleapis.com/cmip6/pangeo-cmip6.csv")
subset = catalog[
    (catalog["variable_id"] == "tas") &
    (catalog["table_id"] == "Amon") &
    (catalog["experiment_id"].isin(["historical", SCENARIO]))
].copy()

keys = ["source_id", "member_id", "grid_label"]
counts = (
    subset.groupby(keys)["experiment_id"]
    .nunique()
    .reset_index(name="n_experiments")
)
available = counts[counts["n_experiments"] == 2]
if available.empty:
    raise ValueError("No matching historical/future datasets found.")

preferred = available[available["source_id"] == PREFERRED_MODEL]
selection = preferred.iloc[0] if not preferred.empty else available.iloc[0]
model = selection["source_id"]
member = selection["member_id"]
grid = selection["grid_label"]
print(model, member, grid)

In [ ]:
fs = gcsfs.GCSFileSystem(token="anon")
records = []

for experiment, first, last in [
    ("historical", START_YEAR, min(2014, END_YEAR)),
    (SCENARIO, max(2015, START_YEAR), END_YEAR)
]:
    if first > last:
        continue

    matches = subset[
        (subset["source_id"] == model) &
        (subset["member_id"] == member) &
        (subset["grid_label"] == grid) &
        (subset["experiment_id"] == experiment)
    ].sort_values("version")

    ds = xr.open_zarr(fs.get_mapper(matches.iloc[-1]["zstore"]), consolidated=True)
    lons = ds["lon"]
    lon = LON % 360 if float(lons.max()) > 180 else LON
    temp = ds["tas"].sel(lat=LAT, lon=lon, method="nearest")
    temp = temp.where(
        (temp.time.dt.year >= first) & (temp.time.dt.year <= last),
        drop=True
    )
    temp = (temp - 273.15).load()

    records.append(pd.DataFrame({
        "year": temp.time.dt.year.values,
        "month": temp.time.dt.month.values,
        "temperature": temp.values
    }))
    ds.close()

data = pd.concat(records, ignore_index=True)
heatmap = data.pivot_table(
    index="year", columns="month", values="temperature", aggfunc="mean"
).reindex(
    index=range(START_YEAR, END_YEAR + 1),
    columns=range(1, 13)
)
heatmap.head()

In [ ]:
temperatures = heatmap.values
if np.isnan(temperatures).all():
    raise ValueError("No temperature data found for this location.")

n_years, n_months = temperatures.shape
theta = np.linspace(0, 2 * np.pi, n_months + 1)
inner_radius = 3
radii = np.linspace(inner_radius, inner_radius + n_years, n_years + 1)

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={"projection": "polar"})
fig.patch.set_facecolor("#faf9f6")
ax.set_facecolor("#faf9f6")

cmap = plt.get_cmap("YlOrRd")
norm = Normalize(vmin=np.nanmin(temperatures), vmax=np.nanmax(temperatures))

for i in range(n_years):
    for j in range(n_months):
        temperature = temperatures[i, j]
        if np.isnan(temperature):
            continue
        ax.bar(
            theta[j],
            height=radii[i + 1] - radii[i],
            width=2 * np.pi / n_months,
            bottom=radii[i],
            color=cmap(norm(temperature)),
            edgecolor="#faf9f6",
            linewidth=0.2,
            align="edge"
        )

ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ax.set_xticks((theta[:-1] + theta[1:]) / 2)
ax.set_xticklabels(months, fontsize=11, fontweight="bold")

year_ticks = np.arange(START_YEAR, END_YEAR + 1, 10)
ax.set_yticks(inner_radius + (year_ticks - START_YEAR) + 0.5)
ax.set_yticklabels(year_ticks, fontsize=9)
ax.set_rlabel_position(0)
ax.grid(False)
ax.spines["polar"].set_visible(False)
ax.text(0, 0, f"SAN DIEGO\n{START_YEAR}–{END_YEAR}",
        ha="center", va="center", fontsize=13, fontweight="bold")

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation="horizontal",
                    fraction=0.04, pad=0.1, shrink=0.65)
cbar.set_label("Average Monthly Temperature (°C)")

fig.suptitle("The Future of Seasons", fontsize=22, fontweight="bold", y=0.98)
fig.text(0.5, 0.04, f"CMIP6 | {model} | {SCENARIO}",
         ha="center", fontsize=10, color="gray")
plt.savefig("circular_temperature_calendar.png", dpi=300, bbox_inches="tight")
plt.show()